# Аналитическое обоснование негативных результатов (без данных)

Чисто модельный расчёт: почему при толстых тканях ρ₂ неидентифицируем, облако решений разваливается, а Δρ₁ ловит ошибку модели. Всё выводится из прямой формулы и её производных — **без экспериментальных данных**. Это мост к МКЭ (после — расчёт по конечно-элементным моделям в MATLAB).

**Что доказываем:**
1. вся обусловленность зависит только от безразмерной глубины η=h/a (§1);
2. чувствительность к ρ₂ падает как ~1/η³ при толстых тканях (§2);
3. неопределённость ρ₂ (Крамер–Рао) взрывается с h — у Георга std(ρ₂)≫ρ₂ (§3);
4. усиление ошибки h→ρ₂ и развал облака (угол кривих → 0) (§4);
5. по одной решётке параметры не разделить; систолический split обусловлен L-формой, но ошибка модели по глубине протекает в Δρ₁ (§5);
6. критический размер L∝h: диапазон 50–140 для Георга ниже порога (§6).

In [ ]:
# @title Модель Z, производные, эластичности (самодостаточно)
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
NT=400
def _terms(r1,r2,h,a,b,N=NT):
    k=(r2-r1)/(r1+r2); d1,d2=a-b,a+b; i=np.arange(1,N+1)
    G1=1/np.sqrt(d1**2+(2*i*h)**2); G2=1/np.sqrt(d2**2+(2*i*h)**2)
    return k,d1,d2,i,G1,G2,G1-G2
def Z(r1,r2,h,a,b,N=NT):
    k,d1,d2,i,G1,G2,dG=_terms(r1,r2,h,a,b,N)
    return r1/np.pi*(1/d1-1/d2)+2*r1/np.pi*np.sum(k**i*dG)
def dZ_dr2(r1,r2,h,a,b,N=NT):
    k,d1,d2,i,G1,G2,dG=_terms(r1,r2,h,a,b,N); T=np.sum(i*k**(i-1)*dG)
    return 4*r1**2/(np.pi*(r1+r2)**2)*T
def dZ_dr1(r1,r2,h,a,b,N=NT):
    k,d1,d2,i,G1,G2,dG=_terms(r1,r2,h,a,b,N); T=np.sum(i*k**(i-1)*dG)
    return Z(r1,r2,h,a,b,N)/r1-4*r1*r2/(np.pi*(r1+r2)**2)*T
def dZ_dh(r1,r2,h,a,b,N=NT):
    k,d1,d2,i,G1,G2,dG=_terms(r1,r2,h,a,b,N); U=np.sum(i**2*k**i*(G1**3-G2**3))
    return -8*r1*h/np.pi*U
def S_r2(r1,r2,h,a,b): return r2/Z(r1,r2,h,a,b)*dZ_dr2(r1,r2,h,a,b)
def S_h(r1,r2,h,a,b):  return h /Z(r1,r2,h,a,b)*dZ_dh(r1,r2,h,a,b)
def ab(L_mm,beta=0.5): a=L_mm/2/1000.0; return a,beta*a

# базовая рабочая точка (физиологичная, не из данных)
R1,R2=6.0,18.0
print("Модель и производные готовы. Рабочая точка ρ1=%.0f, ρ2=%.0f Ом·м." % (R1,R2))

## §1. Всё определяется безразмерной глубиной η = h/a

Безразмерная редукция (см. документ 04): $Z=\dfrac{\rho_1}{a}\,\Phi\!\left(\beta,\ \eta=\tfrac{h}{a},\ k\right)$. Значит **все эластичности и обусловленность зависят только от (β, η, k)** — не от абсолютных h или a по отдельности. Проверим: при фиксированных β,k,η величина $Z\,a/\rho_1$ и $S_{\rho_2}$ инвариантны к масштабу.

In [ ]:
# @title Проверка масштабной инвариантности (Z·a/ρ1 и S_ρ2 зависят только от η,k,β)
print("η     | Z·a/ρ1 (масштаб ×1, ×3)        | S_ρ2 (×1, ×3)")
for eta in [0.3,0.6,1.0]:
    for a_mm,tag in [(50,"×1"),(150,"×3")]:
        a=a_mm/1000.0; b=0.5*a; h=eta*a
        za=Z(R1,R2,h,a,b)*a/R1; s=S_r2(R1,R2,h,a,b)
        if tag=="×1": za1,s1=za,s
    print("%.1f   | %.5f / %.5f (совпали: %s) | %.4f / %.4f" %
          (eta, za1, za, np.isclose(za1,za,rtol=1e-6), s1, s))
print("\nВывод: форма зависимости и чувствительность — функции ТОЛЬКО η=h/a (и β,k). Толщина входит лишь через η.")

## §2. Чувствительность к ρ₂ падает как ~1/η³

$S_{\rho_2}=\dfrac{\rho_2}{Z}\dfrac{\partial Z}{\partial\rho_2}$ — доля сигнала, несущая информацию о лёгком. Аналитически (первый член ряда изображений при больших η): $\Delta G_1\sim \dfrac{ab}{4h^3}$, откуда

$$ S_{\rho_2}\ \sim\ \text{const}\cdot\frac{\rho_1\rho_2}{(\rho_1+\rho_2)^2}\,\beta\left(\frac{a}{h}\right)^3=\frac{\text{const}}{\eta^3}. $$

То есть при толстых тканях чувствительность к ρ₂ исчезает **кубически**. Покажем log-log с наклоном −3.

In [ ]:
# @title S_ρ2(η): коллапс ~1/η³ + положение Ника и Георгия
eta=np.logspace(-1.0,0.8,80)
a=0.035  # фиксируем a, меняем h=η·a (форма от a не зависит, §1)
S=np.array([S_r2(R1,R2,e*a,a,0.5*a) for e in eta])
plt.figure(figsize=(8,5))
plt.loglog(eta,S,lw=2,label="$S_{ρ2}(η)$")
ref=S[40]*(eta/eta[40])**-3; plt.loglog(eta,ref,"k--",lw=1,label="наклон −3 (∝1/η³)")
# диапазоны η=h/a для решёток 50..140 (a=L/2)
for h_mm,name,c in [(15,"Ник",'tab:blue'),(40,"Георгий",'tab:red')]:
    e=[h_mm/(L/2) for L in [50,140]]
    plt.axvspan(min(e),max(e),alpha=0.15,color=c,label="%s: η=%.2f..%.2f"%(name,min(e),max(e)))
plt.xlabel("η = h/a"); plt.ylabel("$S_{ρ2}$ (доля сигнала от лёгкого)")
plt.title("Чувствительность к ρ₂ исчезает с глубиной как ~1/η³"); plt.legend(fontsize=8); plt.grid(True,which="both"); plt.tight_layout(); plt.show()
print("Георгий (толстые ткани) сидит в зоне высокого η -> S_ρ2 в разы меньше, чем у Ника.")

## §3. Неопределённость ρ₂ взрывается с h (граница Крамера–Рао)

h известно (измерено), ищем ρ₂ по набору решёток. Матрица Фишера 2×2 по $(\rho_1,\rho_2)$: $\mathbf M=\sum_k \mathbf g_k\mathbf g_k^\top/\sigma_k^2$, $\mathbf g_k=(\partial_{\rho_1}Z,\partial_{\rho_2}Z)$. Нижняя граница: $\mathrm{std}(\hat\rho_2)=\sqrt{(\mathbf M^{-1})_{22}}$. При относительном шуме $\sigma_k=\varepsilon Z_k$ это чистая функция геометрии и h.

In [ ]:
# @title std(ρ2)/ρ2 vs толщина h — для набора 50..140 мм
EPS=0.01  # относительный шум измерителя, 1%
Ls=np.arange(50,141,10)
def std_rho2(h):
    G=np.array([[dZ_dr1(R1,R2,h,*ab(L)),dZ_dr2(R1,R2,h,*ab(L))] for L in Ls])
    sig=np.array([EPS*Z(R1,R2,h,*ab(L)) for L in Ls])
    M=(G/sig[:,None]).T@(G/sig[:,None])
    return np.sqrt(np.linalg.inv(M)[1,1])
hs=np.linspace(0.005,0.060,60)
sd=np.array([std_rho2(h) for h in hs])
plt.figure(figsize=(8,5))
plt.semilogy(hs*1000, sd/R2*100, lw=2)
for h_mm,name in [(15,"Ник"),(40,"Георгий")]:
    v=std_rho2(h_mm/1000)/R2*100
    plt.scatter([h_mm],[v],zorder=5); plt.annotate("%s: ±%.0f%%"%(name,v),(h_mm,v),fontsize=9)
plt.axhline(100,color="red",ls="--",lw=1,label="неопределённость = 100% ρ2 (предел осмысленности)")
plt.xlabel("толщина h, мм"); plt.ylabel("std($\\hatρ_2$)/ρ2, % (Крамер–Рао, шум 1%)")
plt.title("Неопределённость ρ2 взрывается с толщиной тканей"); plt.legend(); plt.grid(True,which="both"); plt.tight_layout(); plt.show()
print("При h=15 ρ2 определяется; при h=40 std(ρ2) сравнима/больше самого ρ2 -> неидентифицируем (как и в данных: упёрся в границу).")

## §4. Усиление ошибки h→ρ₂ и развал облака (угол кривих → 0)

Перенос ошибки: $\dfrac{\partial\ln\rho_2}{\partial\ln h}\big|_Z=-\dfrac{S_h}{S_{\rho_2}}=1/\mathrm{FoM}$. Геометрия облака: кривые $(\rho_1,\rho_2)$ имеют направление градиента $\phi=\mathrm{atan2}(S_{\rho_2}/\rho_2,\ S_{\rho_1}/\rho_1)$; разброс $\phi$ по решёткам задаёт обусловленность пересечений ($\propto 1/\sin\Delta\phi$).

In [ ]:
# @title FoM(h) и угловой спред кривих по решёткам
def spread_angle(h):
    ang=[np.degrees(np.arctan2(dZ_dr2(R1,R2,h,*ab(L)), dZ_dr1(R1,R2,h,*ab(L)))) for L in Ls]
    return max(ang)-min(ang)
def fom(h,L): s2=S_r2(R1,R2,h,*ab(L)); sh=S_h(R1,R2,h,*ab(L)); return s2/abs(sh)
fig,ax=plt.subplots(1,2,figsize=(14,5))
ax[0].plot(hs*1000,[fom(h,140) for h in hs],lw=2)
ax[0].axhline(1,color="red",ls="--",label="FoM=1 (10% h -> 10% ρ2)")
for h_mm,n in [(15,"Ник"),(40,"Георгий")]: ax[0].scatter([h_mm],[fom(h_mm/1000,140)],zorder=5,label="%s"%n)
ax[0].set_xlabel("h, мм"); ax[0].set_ylabel("FoM = S_ρ2/|S_h| (L=140)"); ax[0].set_title("Сигнал/помеха падает с h"); ax[0].legend(); ax[0].grid(True)
ax[1].plot(hs*1000,[spread_angle(h) for h in hs],lw=2,color="tab:green")
for h_mm,n in [(15,"Ник"),(40,"Георгий")]:
    v=spread_angle(h_mm/1000); ax[1].scatter([h_mm],[v],zorder=5); ax[1].annotate("%s: %.0f°"%(n,v),(h_mm,v),fontsize=9)
ax[1].set_xlabel("h, мм"); ax[1].set_ylabel("разброс направлений кривих, °")
ax[1].set_title("Угол кривих → 0 с толщиной → облако разваливается"); ax[1].grid(True)
plt.tight_layout(); plt.show()
print("Толстые ткани: FoM<1 (ошибка h усиливается) и угол кривих мал (1/sinθ велик) -> облако в разы шире.")

## §5. По одной решётке не разделить; систолический split и протекание ошибки глубины в Δρ₁

Одна решётка: $\Delta Z=g_1\Delta\rho_1+g_2\Delta\rho_2$ — одно уравнение, два неизвестных (ранг 1, бесконечно решений). Набор решёток: split обусловлен **разностью L-форм** $j_1$ (падает), $j_2$ (растёт). НО если истинный пульсовой источник **глубже** однородного полупространства (эффективно $h+\Delta d$ для лёгочного члена), его L-форма иная, и фит на номинальном базисе **сваливает остаток в Δρ₁**. Покажем это аналитически.

In [ ]:
# @title Демонстрация: смещение источника по глубине -> ложное Δρ1
h0=0.025; Dr2_true=0.08    # истинная пульсация лёгкого
Ls=np.arange(50,141,10)
J=np.array([[dZ_dr1(R1,R2,h0,*ab(L)),dZ_dr2(R1,R2,h0,*ab(L))] for L in Ls])  # базис на номинальном h0
print("Δd, мм | восстановл. Δρ1 (ложное) | Δρ2 | трактовка")
print("-"*64)
for dd in [0,5,10,20]:
    # "истина": источник лёгкого эффективно глубже на Δd (другая L-форма)
    dZ_true=np.array([dZ_dr2(R1,R2,h0+dd/1000,*ab(L))*Dr2_true for L in Ls])
    sol,*_=np.linalg.lstsq(J,dZ_true,rcond=None)
    print("  %2d   |      %+.4f Ом·м       | %.4f | %s" %
          (dd, sol[0], sol[1], "Δρ1=0 (нет ошибки)" if dd==0 else "ошибка глубины -> ложное Δρ1≠0"))
print("\nВывод: при ВЕРНОЙ модели (Δd=0) Δρ1=0 точно. Любое несоответствие глубины источника")
print("плоской двуслойной модели проецируется в Δρ1 -> это систематический предел (нужен МКЭ).")

## §6. Критический размер L∝h: диапазон 50–140 для Георга ниже порога

Порог идентифицируемости $S_{\rho_2}\ge S^\ast$ сводится к условию на η: $L_{\min}=2h/\eta^\ast\propto h$. Покажем, какой размер нужен для порога и что у Георга он за пределами 140 мм.

In [ ]:
# @title L для порога S_ρ2 vs толщина
S_STAR=0.25
def L_for_threshold(h):
    Lg=np.linspace(40,400,500)
    s=np.array([S_r2(R1,R2,h,*ab(L)) for L in Lg])
    idx=np.where(s>=S_STAR)[0]
    return Lg[idx[0]] if len(idx) else np.nan
hs2=np.linspace(0.005,0.060,40)
Lneed=np.array([L_for_threshold(h) for h in hs2])
plt.figure(figsize=(8,5))
plt.plot(hs2*1000,Lneed,lw=2,label="L для $S_{ρ2}≥%.2f$"%S_STAR)
plt.axhspan(50,140,alpha=0.15,color="green",label="доступный диапазон 50–140 мм")
for h_mm,n in [(15,"Ник"),(40,"Георгий")]:
    v=L_for_threshold(h_mm/1000); plt.scatter([h_mm],[v],zorder=5); plt.annotate("%s: нужно %.0f мм"%(n,v),(h_mm,v),fontsize=9)
plt.xlabel("толщина h, мм"); plt.ylabel("необходимый размер L, мм")
plt.title("Критический размер растёт линейно с h (L∝h)"); plt.legend(); plt.grid(True); plt.tight_layout(); plt.show()
print("Ник: порог достигается внутри 50–140. Георгий: нужно L много больше 140 -> текущими решётками ρ2 не взять.")

## §7. Выводы и мост к МКЭ

**Все негативные результаты — следствия двух фактов:**
1. $S_{\rho_2}\sim1/\eta^3$ — чувствительность к лёгкому исчезает кубически с относительной глубиной. Отсюда: взрыв std(ρ₂) с h (§3), падение FoM и развал облака (§4), критический $L\propto h$ за пределами доступного для толстых тканей (§6).
2. Двухпараметрическая плоская модель **рангово бедна** для глубокого/структурированного источника: одна решётка не разделяет (§5), а несоответствие глубины проецируется в Δρ₁ — систематический предел, не убираемый статистикой.

**Что снимает МКЭ по геометрии КТ (следующий шаг, MATLAB):**
- **глубинно-разрешённый источник** — пульсация на правильной глубине (сердце/сосуды/лёгкое) убирает протекание в Δρ₁ (§5);
- **реальная геометрия** (рёбра, кривизна) — даёт верхнюю границу $L_{\max}$ и позволяет крупные базы, возвращающие $S_{\rho_2}$ при толстых тканях (§2,§6);
- **проверка плоской формулы**: сравнение $Z_{\text{МКЭ}}(L)$ с аналитической $Z(L)$ количественно очерчивает зону применимости.

Эти аналитические зависимости (S_ρ2(η), std(ρ2)(h), FoM, L(h)) — эталон, с которым будем сверять МКЭ.